# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

#%pip install docling markdown-it-py
#%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Document
import glob

# In our example above docling step produces markdown of all the pdf files in the document_collection
with open(glob.glob(f"{data_dir}/*.md")[0], "r") as f:
    text = f.read()

In [ ]:
# Step 4: Text Chunking and Dataset Creation

from markdown_it import MarkdownIt
from typing import List
import datasets


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    # Initialize markdown parser to understand document structure
    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments to preserve markdown structure
    # This ensures we don't split in the middle of headings, lists, etc.
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Split blocks into chunks with overlap to maintain context continuity
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                # Emit a complete chunk
                chunks.append(" ".join(current_words))
                # Prepare next buffer with overlap from the end of this chunk
                # This ensures context continuity between chunks
                current_words = current_words[-overlap:] if overlap > 0 else []

    # Add any remaining words as the final chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


chunks = chunk_markdown(text, max_tokens=5000, overlap=1000)


# Prepare seed data for the SDG-Hub knowledge pipeline.
#
# The seed data requires the following fields:
#   - document_outline: A concise title or summary that accurately represents the entire document.
#     For documents covering multiple themes, consider providing multiple outlines (one per section).
#   - icl_document: A representative sample extract from the document. This may include tables, code snippets, definitions, etc.
#   - icl_query_1, icl_query_2, icl_query_3: Three questions based on the icl_document sample.
#   - domain: The domain or subject area of the document.
#
# The code below creates a HuggingFace Dataset from the document chunks,
# then maps the required ICL fields to each entry, and finally saves the result as a JSONL file.

seed_data = datasets.Dataset.from_dict({"document": chunks})

import os
import json
from litellm import completion

# 1. Pega um chunk longo o suficiente (mais de 100 caracteres) como nossa Amostra "Ouro" (Gold Standard)
sample_text = next((chunk for chunk in chunks if len(chunk) > 100), chunks[0] if chunks else "")

# 2. Configuração do Modelo LiteLLM (As mesmas que você colocou no .env)
# Atenção: Ajuste isso se o seu VLLM_MODEL for diferente!
model_name = os.getenv("VLLM_MODEL", "hosted_vllm/openai/RedHatAI/gpt-oss-20b")
api_base = os.getenv("VLLM_API_BASE", "http://inference-server-ai-inference-server-ocp.apps.cluster-lb89q.lb89q.sandbox857.opentlc.com/v1")
api_key = os.getenv("VLLM_API_KEY", "EMPTY")

print(f"Gerando dados ICL dinamicamente baseados no documento usando o modelo: {model_name}...")

# 3. Prompt sistêmico para instruir o LLM a criar o ICL
prompt = f"""
Você é um especialista em treinamento de LLMs. Analise o seguinte documento e crie exatamente 3 perguntas 
valiosas e desafiadoras que possam ser feitas sobre esse texto.
Responda APENAS com um objeto JSON válido contendo as seguintes chaves: 
"document_outline", "icl_query_1", "icl_query_2", "icl_query_3", "domain"

Texto do Documento:
{sample_text[:1500]} # Limitando o tamanho se o chunk for muito grande

Formato JSON esperado:
{{
  "document_outline": "Resumo de 1 frase descrevendo o documento.",
  "icl_query_1": "Sua pergunta numero 1?",
  "icl_query_2": "Sua pergunta numero 2?",
  "icl_query_3": "Sua pergunta numero 3?",
  "domain": "Recursos Humanos / Feriados"
}}
"""

try:
    # 4. Chamada de LLM para gerar o ICL dinâmico
    response = completion(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        api_base=api_base,
        api_key=api_key,
        temperature=0.3,
        response_format={"type": "json_object"}
    )
    
    # 5. Processamento e Limpeza da resposta
    llm_output = response.choices[0].message.content
    generated_data = json.loads(llm_output)
    
    # Montando o dicionário final com o 'icl_document' usando a própria amostra original
    icl = {
        "document_outline": generated_data.get("document_outline", "Feriados e Emendas de 2026"),
        "icl_document": sample_text,
        "icl_query_1": generated_data.get("icl_query_1", "Pergunta 1?"),
        "icl_query_2": generated_data.get("icl_query_2", "Pergunta 2?"),
        "icl_query_3": generated_data.get("icl_query_3", "Pergunta 3?"),
        "domain": generated_data.get("domain", "Corporativo")
    }
    
    print("\n--- ICL Gerado com Sucesso! ---")
    print(json.dumps(icl, indent=2, ensure_ascii=False))

except Exception as e:
    print(f"Erro ao gerar ICL, usando template de fallback (Erro: {e})")
    # Um fallback seguro caso o LLM falhe
    icl = {
        "document_outline": "Documento Corporativo",
        "icl_document": sample_text,
        "icl_query_1": "Qual é o tema principal deste texto?",
        "icl_query_2": "Quais regras são descritas no documento?",
        "icl_query_3": "Para quem este documento é destinado?",
        "domain": "Corporativo"
    }

# 6. Aplica o dicionário dinâmico no dataset HuggingFace
seed_data = seed_data.map(lambda x: icl)

# Salva o resultado final pronto para o próximo Notebook!
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)
print("\nSalvo com sucesso no seed_data.jsonl!")


### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook